# Titanic Survival Classification with Supervised Machine Learning

**Original project:** WS 2019/2020  
**Portfolio version:** reconstructed from the surviving 2020 code export and group report

This notebook reconstructs the supervised machine-learning workflow used in the original
Data Mining group project, *Machine Learning from Disaster: Predicting Survival Rate*.

The project compares three classifiers:

- k-Nearest Neighbors (KNN)
- Decision Tree
- Random Forest

It also includes exploratory analysis, missing-value handling, categorical encoding,
train/test splitting, model evaluation, feature importance, and prediction on the
Kaggle Titanic test set.

> **Reconstruction note:** the original `.ipynb` file is no longer available.
> This portfolio notebook was rebuilt from the code preserved in the 2020 PDF export
> and the accompanying project report. Historical results reported in those two sources
> are documented separately below because they represent slightly different experiment versions.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

## 2. Load the Training Dataset

Download the Kaggle Titanic competition files and place `train.csv` and `test.csv`
in the repository's `data/` directory.

In [ ]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"

titanic = pd.read_csv(train_path)
titanic.head(12)

## 3. Initial Data Inspection

In [ ]:
print("Shape:", titanic.shape)
titanic.describe(include="all")

In [ ]:
print("Missing values before cleaning:")
print(titanic.isna().sum())

The original project identified missing values in `Age`, `Cabin`, and `Embarked`.
The historical workflow filled missing `Age` values with the mean, removed the
`Cabin` column, removed records with missing `Embarked`, and excluded columns that
were not used as predictors.

## 4. Exploratory Data Analysis

In [ ]:
sns.countplot(data=titanic, x="Survived")
plt.title("Titanic Passenger Survival Counts")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, col in zip(axes.flat, ["Pclass", "Sex", "Parch", "Embarked"]):
    sns.countplot(data=titanic, x=col, hue="Survived", ax=ax)
    ax.set_title(f"Survival by {col}")

plt.tight_layout()
plt.show()

In [ ]:
survival_by_sex = titanic.groupby("Sex")["Survived"].mean()
survival_by_sex

In [ ]:
survival_by_class_and_sex = titanic.pivot_table(
    values="Survived",
    index="Sex",
    columns="Pclass",
    aggfunc="mean"
)
survival_by_class_and_sex

## 5. Data Cleaning and Feature Preparation

In [ ]:
data = titanic.copy()

# Historical project approach: mean imputation for Age
data["Age"] = data["Age"].fillna(data["Age"].mean())

# Drop columns not used by the original model
data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Remove the small number of rows with missing Embarked values
data = data.dropna(subset=["Embarked"]).copy()

print("Remaining shape:", data.shape)
print(data.isna().sum())

In [ ]:
# Encode categorical variables.
# The original project used LabelEncoder for Sex and Embarked.
sex_encoder = LabelEncoder()
embarked_encoder = LabelEncoder()

data["Sex"] = sex_encoder.fit_transform(data["Sex"])
data["Embarked"] = embarked_encoder.fit_transform(data["Embarked"])

data.dtypes

## 6. Define Predictors and Target

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = data[features]
y = data["Survived"]

X.head()

## 7. Train/Test Split

The historical project used an 80/20 train/test split. The surviving code export
used `random_state=0`, while the written report describes `random_state=1`.
This reconstructed notebook uses `random_state=0` to stay closest to the surviving code.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 8. Train the Three Supervised Classifiers

The surviving code export used:

- KNN: `n_neighbors=5`, Minkowski metric with `p=2`
- Decision Tree: entropy criterion
- Random Forest: 68 trees, entropy criterion

Those settings are preserved here. The written report describes a slightly different
experiment version; its historical settings and results are documented later.

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="minkowski",
    p=2
)

decision_tree = DecisionTreeClassifier(
    criterion="entropy",
    random_state=1
)

random_forest = RandomForestClassifier(
    n_estimators=68,
    criterion="entropy",
    random_state=1
)

models = {
    "KNN": knn,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest
}

for model in models.values():
    model.fit(X_train, y_train)

## 9. Training Accuracy

In [ ]:
for name, model in models.items():
    print(f"{name} training accuracy: {model.score(X_train, y_train):.4f}")

## 10. Test-Set Evaluation

In [ ]:
evaluation_rows = []

for name, model in models.items():
    predictions = model.predict(X_test)

    evaluation_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
    })

evaluation = pd.DataFrame(evaluation_rows).set_index("Model")
evaluation

In [ ]:
evaluation.plot(kind="bar", figsize=(9, 5))
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Confusion Matrices

In [ ]:
for name, model in models.items():
    predictions = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, predictions)
    plt.title(f"{name} Confusion Matrix")
    plt.show()

## 12. Decision-Tree Feature Importance

In [ ]:
feature_importance = pd.Series(
    decision_tree.feature_importances_,
    index=features
).sort_values(ascending=False)

feature_importance

In [ ]:
feature_importance.plot(kind="bar", figsize=(8, 5))
plt.ylabel("Importance")
plt.title("Decision Tree Feature Importance")
plt.tight_layout()
plt.show()

## 13. Decision-Tree Visualization

In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(
    decision_tree,
    feature_names=features,
    class_names=["Did Not Survive", "Survived"],
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=8
)
plt.title("Decision Tree — First Four Levels")
plt.show()

## 14. Example Passenger Prediction

The original notebook included a small "Could you have survived the Titanic?"
demonstration using a manually entered passenger feature vector.

For a reconstructed portfolio notebook, the example below uses the same feature order:
`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`.

In [ ]:
# Example corresponding to the surviving 2020 notebook:
# Pclass=3, Sex=male, Age=34.5, SibSp=0, Parch=0, Fare=7.8292, Embarked=Q

example_passenger = pd.DataFrame([{
    "Pclass": 3,
    "Sex": sex_encoder.transform(["male"])[0],
    "Age": 34.5,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.8292,
    "Embarked": embarked_encoder.transform(["Q"])[0]
}])

example_prediction = decision_tree.predict(example_passenger)[0]

if example_prediction == 1:
    print("Model prediction: survived")
else:
    print("Model prediction: did not survive")

## 15. Predict the Kaggle Titanic Test Set

In [ ]:
kaggle_test = pd.read_csv(test_path)
passenger_ids = kaggle_test["PassengerId"].copy()

# Apply the same historical feature preparation
kaggle_test["Age"] = kaggle_test["Age"].fillna(kaggle_test["Age"].mean())
kaggle_test["Fare"] = kaggle_test["Fare"].fillna(kaggle_test["Fare"].mean())

kaggle_test["Sex"] = sex_encoder.transform(kaggle_test["Sex"])

# Handle any unseen/missing Embarked values conservatively
kaggle_test["Embarked"] = kaggle_test["Embarked"].fillna("S")
kaggle_test["Embarked"] = embarked_encoder.transform(kaggle_test["Embarked"])

test_features = kaggle_test[features]
kaggle_predictions = decision_tree.predict(test_features)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": kaggle_predictions
})

submission.head()

In [ ]:
submission.to_csv("Titanic_Predictions.csv", index=False)
print("Saved Titanic_Predictions.csv")

## 16. Historical Results from the 2020 Project

Two surviving sources contain slightly different experiment versions.

### Written group report

The report documents approximately:

| Model | Accuracy | Precision | Recall |
|---|---:|---:|---:|
| KNN | 75% | 72% | 64% |
| Decision Tree | 80% | 79% | 71% |
| Random Forest | 81% | 82% | 68% |

The report states that Random Forest achieved the highest accuracy, while Decision Tree
was retained as the main model for the real-life scenario and Kaggle test predictions.

### Surviving notebook/code export

A separate saved run in the code-export PDF reports approximately:

| Model | Accuracy | Precision | Recall |
|---|---:|---:|---:|
| KNN | 69.1% | 62.9% | 60.3% |
| Decision Tree | 75.3% | 71.6% | 65.8% |
| Random Forest | 75.8% | 74.2% | 63.0% |

Because the parameter settings and random-state choices differ between the report and
the code export, these historical values should not be treated as a single reproducible run.

This reconstructed notebook therefore computes its own metrics when executed and keeps
the historical values only for archival context.

## 17. Portfolio and Collaboration Note

This was a group project completed during the Master's Data Mining course in WS 2019/2020.

Original group members:

- Gerta Mata
- Nafiu Ikeoluwa Hammed
- Onassis Sowah Anyetei

The original report's workload table identifies Hammed's contributions across theoretical
work, data cleansing, train/test splitting and fitting, algorithm comparison, the real-life
prediction scenario, and conclusions.

This GitHub notebook is a later portfolio reconstruction from the surviving project materials.
It does not claim sole authorship of the original group project.